# Tome 2, Chapter 10: Galois Theory (Python companion)

Exercises first, solutions at the end. We compute a Galois correspondence by brute force, ask SymPy for Galois groups, watch Dedekind's theorem at work (the factorisations of a polynomial modulo primes reveal its Galois group, and the frequencies match the group's cycle types), climb Gauss' tower for the $17$-gon numerically, and meet the *casus irreducibilis* of Cardano's formula.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import cmath
from collections import Counter
from fractions import Fraction
import sympy
from sympy import symbols, Poly, sqrt, factor_list, primerange, discriminant, minimal_polynomial
from sympy.polys.numberfields.galoisgroups import galois_group
import matplotlib.pyplot as plt
X = symbols('X')

## Exercises

**P10.1 (★) A Galois correspondence by brute force.** For $L=\Q(\sqrt2,\sqrt3,\sqrt5)$, represent each automorphism by its signs $(\pm1,\pm1,\pm1)$. Enumerate all subgroups of this group of order $8$; for each, find the products $\sqrt{d}$, $d\mid30$, that it fixes, and check that the number of subgroups of each order matches Exercise 10.1.

In [ ]:
# Your code here


**P10.2 (★) Galois groups with SymPy.** Compute the Galois groups of $X^3-2$, $X^3-3X+1$, $X^4-2$, $X^4+1$, $X^4-10X^2+1$, $X^5-4X+2$, $X^5-X-1$ and $X^5-5X+12$, together with their discriminants and numbers of real roots. Which are solvable by radicals? Then test $2000$ random quintics $X^5+aX+b$ with $|a|,|b|\le 20$: how often is the group $\mathfrak S_5$?

In [ ]:
# Your code here


**P10.3 (★★) Dedekind's theorem as an experiment.** For $f=X^5-4X+2$, factor $f$ modulo each prime $p<20000$ not dividing the discriminant, and record the degrees of the factors (a *cycle type*). Compare the frequencies with the proportions of elements of each cycle type in $\mathfrak S_5$. Do the same for $X^4-2$ (group $D_4$) and $X^4+1$ (group $V_4$).

In [ ]:
# Your code here


**P10.4 (★★) Gauss' tower for the $17$-gon.** Compute numerically the periods of Exercise 10.4 and the next two floors (sums over the cosets of the subgroups of order $4$ and $2$ of $(\Z/17\Z)^*=\langle3\rangle$), check that each floor is obtained from the previous one by a quadratic equation, and recover $\cos\frac{2\pi}{17}$ from square roots only.

In [ ]:
# Your code here


**P10.5 (★) Casus irreducibilis.** Apply Cardano's formula (Exercise 10.7) to $X^3-3X+1$, whose three roots are real. What happens under the square root? Check that the formula still gives the roots, provided one computes with complex numbers.

In [ ]:
# Your code here


---
## Solutions

### P10.1

In [ ]:
G = list(itertools.product((1, -1), repeat=3))          # signs of sqrt2, sqrt3, sqrt5
mul = lambda g, h: tuple(a * b for a, b in zip(g, h))
def closure(gens):
    S = {(1, 1, 1)}
    while True:
        new = S | {mul(a, b) for a in S for b in S | set(gens)}
        if new == S: return frozenset(S)
        S = new
subgroups = {closure(c) for k in range(4) for c in itertools.combinations(G, k)}
primes = [2, 3, 5]
def fixed_roots(H):
    out = []
    for mask in range(1, 8):
        d = math.prod(p for i, p in enumerate(primes) if mask >> i & 1)
        if all(math.prod(h[i] for i in range(3) if mask >> i & 1) == 1 for h in H):
            out.append(f'sqrt{d}')
    return out
by_order = Counter(len(H) for H in subgroups)
print('number of subgroups by order:', dict(sorted(by_order.items())))
for H in sorted(subgroups, key=len):
    if len(H) == 4:
        print('order 4, fixes', fixed_roots(H))

There are $1+7+7+1=16$ subgroups. The $7$ subgroups of order $4$ (index $2$) each fix exactly one $\sqrt d$, $d\in\{2,3,5,6,10,15,30\}$: the $7$ quadratic fields of Exercise 10.1. The $7$ subgroups of order $2$ fix three square roots each, generating fields of degree $4$.

### P10.2

In [ ]:
tests = [X**3 - 2, X**3 - 3*X + 1, X**4 - 2, X**4 + 1, X**4 - 10*X**2 + 1, X**5 - 4*X + 2, X**5 - X - 1, X**5 - 5*X + 12]
solvable_names = {'S3': True, 'A3': True, 'S4': True, 'A4': True, 'D4': True, 'V': True, 'C4': True,
                  'S5': False, 'A5': False, 'M20': True, 'D5': True, 'C5': True}
for f in tests:
    P = Poly(f, X)
    name = galois_group(P, by_name=True)[0].name
    print(f'{str(f):18s} group {name:4s} disc {int(discriminant(P)):>9d}  real roots {P.count_roots()}  solvable: {solvable_names[name]}')

random_counts = Counter()
import random
random.seed(1)
tried = 0
while tried < 2000:
    a, b = random.randint(-20, 20), random.randint(-20, 20)
    P = Poly(X**5 + a*X + b, X)
    if not P.is_irreducible:
        continue
    tried += 1
    random_counts[galois_group(P, by_name=True)[0].name] += 1
print(dict(random_counts))

$X^5-X-1$ also has group $\mathfrak S_5$, although it has only one real root (so the proof of Theorem 10.14 does not apply to it; SymPy uses resolvents instead). $X^5-5X+12$ has the solvable group $D_5$ of order $10$: its roots *can* be written with radicals, which Abel and Galois never claimed impossible for particular quintics. Among irreducible random trinomials, almost all have group $\mathfrak S_5$ (Hilbert's irreducibility theorem and van der Waerden's theorem make this precise); the exceptions are rare and special.

### P10.3

In [ ]:
S5_types = Counter()
for perm in itertools.permutations(range(5)):
    seen, cyc = set(), []
    for i in range(5):
        if i not in seen:
            j, l = i, 0
            while j not in seen:
                seen.add(j); j = perm[j]; l += 1
            cyc.append(l)
    S5_types[tuple(sorted(cyc))] += 1

def frobenius_stats(f, pmax=20000):
    P = Poly(f, X); disc = int(discriminant(P))
    stats = Counter(); n = 0
    for p in primerange(3, pmax):
        if disc % p == 0: continue
        degs = tuple(sorted(sympy.degree(g, X) * e for g, e in factor_list(f, modulus=p)[1]))
        stats[degs] += 1; n += 1
    return stats, n

stats, n = frobenius_stats(X**5 - 4*X + 2)
print(f'X^5-4X+2 over {n} primes:')
for t in sorted(S5_types, key=lambda t: -S5_types[t]):
    print(f'  {str(t):18s} observed {stats[t] / n:.3f}   proportion in S5 {S5_types[t] / 120:.3f}')
for f, expected in [(X**4 - 2, {(1, 1, 1, 1): 1/8, (1, 1, 2): 2/8, (2, 2): 3/8, (4,): 2/8}),
                    (X**4 + 1, {(1, 1, 1, 1): 1/4, (2, 2): 3/4})]:
    stats, n = frobenius_stats(f)
    print(f, {t: round(stats[t] / n, 3) for t in stats}, ' expected', {t: round(v, 3) for t, v in expected.items()})

The observed frequencies match the proportions of cycle types in the Galois group: this is the *Chebotarev density theorem* (1922), the quantitative form of Dedekind's theorem quoted in the remark of Section 10.4. For $X^4-2$, the $D_4$ inside $\mathfrak S_4$ contains the identity, two transpositions (reflections through a diagonal of the square of roots), three double transpositions and two $4$-cycles, and the frequencies $1/8, 2/8, 3/8, 2/8$ appear; no prime makes $X^4-2$ split as a $3$-cycle times a fixed point, because $D_4$ has no element of order $3$. For $X^4+1$ only the identity and double transpositions occur, so $X^4+1$ is never irreducible modulo $p$ (Exercise 6.8). Read backwards, factoring modulo a few hundred primes is how computer algebra systems *guess* Galois groups quickly.

### P10.4

In [ ]:
p, g = 17, 3
z = cmath.exp(2j * math.pi / p)
powers = [pow(g, k, p) for k in range(16)]
def periods(m):
    # sums over the cosets of the subgroup of order 16/m, i.e. exponents congruent mod m
    return [sum(z ** powers[k] for k in range(j, 16, m)).real for j in range(m)]
floors = {m: periods(m) for m in (2, 4, 8)}          # floor 8: the numbers 2cos(2 pi a/17)
for m, ps in floors.items():
    print(m, 'periods:', [round(x, 6) for x in ps])

# each pair of periods on one floor is the pair of roots of a quadratic whose coefficients are periods of the previous floor
for m in (4, 8):
    for j in range(m // 2):
        a, b = floors[m][j], floors[m][j + m // 2]
        print(f'floor {m}: pair ({a:.6f}, {b:.6f})  sum {a + b:.6f}  product {a * b:.6f}')

# Gauss' formula, with square roots only
s17 = math.sqrt(17)
c = (-1 + s17 + math.sqrt(34 - 2 * s17) + 2 * math.sqrt(17 + 3 * s17 - math.sqrt(34 - 2 * s17) - 2 * math.sqrt(34 + 2 * s17))) / 16
print('Gauss formula :', c)
print('cos(2pi/17)   :', math.cos(2 * math.pi / 17))

On each floor the sum and the product of each pair are (up to rounding) sums of periods of the floor below: for instance on floor $4$ the sums are $\eta_0=1.5615$ and $\eta_1=-2.5615$, and the products are $-1$. So every floor is reached by square roots, as Theorem 10.10 predicts from the chain of subgroups of $\Z/16\Z$. Unwinding the formulas gives Gauss' famous expression
$$16\cos\frac{2\pi}{17}=-1+\sqrt{17}+\sqrt{34-2\sqrt{17}}+2\sqrt{17+3\sqrt{17}-\sqrt{34-2\sqrt{17}}-2\sqrt{34+2\sqrt{17}}},$$
which the last two lines check numerically.

### P10.5

In [ ]:
pp, qq = -3, 1
D = qq**2 / 4 + pp**3 / 27
print('under the square root:', D)
w = cmath.sqrt(D)
u = (-qq / 2 + w) ** (1 / 3)                       # principal complex cube root
roots_cardano = []
for k in range(3):
    uk = u * cmath.exp(2j * math.pi * k / 3)
    vk = -pp / (3 * uk)                              # the cube roots must have product -p/3
    roots_cardano.append(uk + vk)
print('Cardano :', [complex(round(r.real, 12), round(r.imag, 12)) for r in roots_cardano])
print('actual  :', sorted(2 * math.cos(2 * math.pi * k / 9) for k in (1, 2, 4)))

The quantity under the square root is $\frac14-1=-\frac34<0$, although all three roots are real: Cardano's formula goes through the complex numbers and back. This *casus irreducibilis* is unavoidable: one can prove with Galois theory that the real roots of an irreducible cubic with three real roots can never be written with *real* radicals. It is historically how complex numbers entered mathematics (Bombelli, 1572), long before anyone accepted them as numbers.